### SQL Authoring Note

Use `%%sql` cells for SQL-first workflows in this chapter.

Tips:
- Use the toolbar **Format SQL** action for query formatting.
- Keep DDL/DML in dedicated SQL cells and reserve Python cells for orchestration.


In [3]:
%%sql
CREATE DATABASE IF NOT EXISTS bronze;

++
||
++
++

In [4]:
%%sql
SHOW TABLES IN bronze;

namespace,tableName,isTemporary


In [5]:
%%sql 
CREATE DATABASE IF NOT EXISTS silver;

++
||
++
++

In [6]:
%%sql
SHOW TABLES IN silver;

namespace,tableName,isTemporary


In [7]:
%%sql
CREATE DATABASE IF NOT EXISTS gold;

++
||
++
++

In [8]:
%%sql
SHOW TABLES IN gold;

namespace,tableName,isTemporary


In [9]:
%%sql
SHOW DATABASES;

namespace
bronze
gold
silver


In [10]:
%%sql
CREATE TABLE
  IF NOT EXISTS bronze.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (days (created_at)) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for user information'
  );

++
||
++
++

In [11]:
%%sql
CREATE TABLE
  IF NOT EXISTS bronze.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7, 2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (category) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
  );

++
||
++
++

In [12]:
%%sql
CREATE TABLE
  IF NOT EXISTS bronze.purchases (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12, 2),
    created_at TIMESTAMP,
    updated_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (days (created_at)) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
  );

++
||
++
++

In [13]:
%%sql
CREATE TABLE
  IF NOT EXISTS bronze.pageviews (
    user_id BIGINT,
    url STRING,
    channel STRING,
    received_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (days (received_at)) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
  );

++
||
++
++

In [14]:
%%sql
DESCRIBE TABLE bronze.users;

col_name,data_type,comment
id,bigint,None
first_name,string,None
last_name,string,None
email,string,None
created_at,timestamp,None
updated_at,timestamp,None
,,
# Partitioning,,
Part 0,days(created_at),


In [15]:
%%sql
CREATE TABLE
  IF NOT EXISTS silver.users (
    id BIGINT,
    first_name STRING,
    last_name STRING,
    email STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    valid_email BOOLEAN,
    full_name STRING
  ) USING iceberg PARTITIONED BY (days (created_at)) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated dimension table for user information'
  );

++
||
++
++

In [16]:
%%sql
CREATE TABLE
  IF NOT EXISTS silver.items (
    id BIGINT,
    name STRING,
    category STRING,
    price DECIMAL(7, 2),
    inventory INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (category) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Dimension table for product items'
  );

++
||
++
++

In [17]:
%%sql
CREATE TABLE
  IF NOT EXISTS silver.purchases_enriched (
    id BIGINT,
    user_id BIGINT,
    item_id BIGINT,
    quantity INT,
    purchase_price DECIMAL(12, 2),
    total_price DECIMAL(14, 2),
    user_email STRING,
    item_name STRING,
    item_category STRING,
    purchase_date DATE,
    purchase_hour INT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (days (created_at)) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Validated and enriched fact table for purchase transactions'
  );

++
||
++
++

In [19]:
%%sql
DESCRIBE TABLE silver.purchases_enriched;

col_name,data_type,comment
id,bigint,None
user_id,bigint,None
item_id,bigint,None
quantity,int,None
purchase_price,"decimal(12,2)",None
total_price,"decimal(14,2)",None
user_email,string,None
item_name,string,None
item_category,string,None
created_at,timestamp,None


In [20]:
%%sql
CREATE TABLE
  IF NOT EXISTS silver.pageviews_by_items (
    user_id BIGINT,
    item_id BIGINT,
    page STRING,
    item_name STRING,
    item_category STRING,
    channel STRING,
    received_at TIMESTAMP
  ) USING iceberg PARTITIONED BY (days (received_at)) TBLPROPERTIES (
    'format-version' = '2',
    'comment' = 'Fact table for purchase transactions'
  );

++
||
++
++

In [21]:
%%sql
SHOW TBLPROPERTIES silver.pageviews_by_items;

key,value
current-snapshot-id,none
format,iceberg/parquet
format-version,2
write.parquet.compression-codec,zstd


In [22]:
%%sql
DESCRIBE TABLE EXTENDED silver.pageviews_by_items;

col_name,data_type,comment
user_id,bigint,None
item_id,bigint,None
page,string,None
item_name,string,None
item_category,string,None
channel,string,None
received_at,timestamp,None
,,
# Partitioning,,
Part 0,days(received_at),
